# Calculadora de Juros Compostos
## Simulador modular de investimentos

Organizado em 4 blocos:
1. **Variáveis / Configurações** — parâmetros de entrada
2. **Funções** — fórmulas matemáticas encapsuladas
3. **Execução** — processamento dos dados
4. **Resultados e Visualização** — exibição e gráficos

---
## Bloco 1 — Variáveis / Configurações

Defina aqui os parâmetros da simulação.

In [ ]:
# =========================================================================
# PARAMETROS DE ENTRADA
# =========================================================================

CAPITAL_INICIAL = 0.0          # Saldo inicial (R$)
TAXA_MENSAL_PCT = 0.83         # Taxa de juros mensal (%)
META = 1_000_000               # Valor desejado (R$)
APORTE_MENSAL = 500.0          # Aporte mensal fixo (R$)
PRAZO_MESES = 360              # Prazo maximo em meses (30 anos)
MES_INICIAL = 0                # Mes de partida (0 = inicio)

In [ ]:
# =========================================================================
# CONFIGURACOES AVANCADAS
# =========================================================================

ESTRATEGIA = "aporte_depois"   # 'aporte_depois' ou 'aporte_antes'
EXPORTAR_CSV = True            # Exportar simulacao para .csv ao final
ARQUIVO_CSV = "simulacao_mes_a_mes.csv"

---
## Bloco 2 — Funções

Fórmulas matemáticas encapsuladas para o cálculo dos juros compostos.

In [ ]:
# =========================================================================
# FUNCOES MATEMATICAS
# =========================================================================

def calcular_montante_futuro(pv: float, taxa: float, n: int) -> float:
    """Valor Futuro (VF) = PV * (1 + i)^n"""
    return pv * (1 + taxa) ** n


def calcular_valor_presente(fv: float, taxa: float, n: int) -> float:
    """Valor Presente (VP) = FV / (1 + i)^n"""
    return fv / (1 + taxa) ** n


def calcular_aporte_necessario(alvo: float, taxa: float, n: int) -> float:
    """Aporte mensal necessario para atingir o alvo em n meses."""
    if taxa <= 0:
        return alvo / n if n > 0 else float("inf")
    return alvo * taxa / ((1 + taxa) ** n - 1)


def calcular_taxa_equivalente(taxa_mensal: float) -> dict:
    """Converte a taxa mensal para outras periodicidades."""
    return {
        "mensal": taxa_mensal,
        "anual": (1 + taxa_mensal) ** 12 - 1,
        "diaria": (1 + taxa_mensal) ** (1 / 30) - 1,
        "semestral": (1 + taxa_mensal) ** 6 - 1,
    }


def projetar_saldo(capital: float, aporte: float, taxa: float,
                    meses: int) -> list[dict]:
    """Projeta mes a mes o saldo, rendimento e total investido."""
    saldo = capital
    total_investido = capital
    registros = []

    for mes in range(1, meses + 1):
        rendimento = saldo * taxa
        saldo = saldo * (1 + taxa) + aporte
        total_investido += aporte
        registros.append({
            "Mes": mes,
            "Ano": round(mes / 12, 2),
            "Aporte": aporte,
            "Rendimento": round(rendimento, 2),
            "Saldo": round(saldo, 2),
            "Total_Investido": round(total_investido, 2),
        })

    return registros


def extrair_resumo(registros: list[dict]) -> dict:
    """Extrai indicadores resumidos da projecao."""
    if not registros:
        return {}
    ultimo = registros[-1]
    total_inv = ultimo["Total_Investido"]
    total_juros = round(ultimo["Saldo"] - total_inv, 2)
    pct_juros = total_juros / ultimo["Saldo"] * 100 if ultimo["Saldo"] > 0 else 0
    return {
        "saldo_final": ultimo["Saldo"],
        "total_investido": total_inv,
        "total_juros": total_juros,
        "pct_juros": round(pct_juros, 2),
        "meses": ultimo["Mes"],
        "anos": round(ultimo["Mes"] / 12, 1),
    }

---
## Bloco 3 — Execução

Chamada das funções utilizando os parâmetros definidos no Bloco 1.

In [ ]:
# =========================================================================
# PREPARACAO
# =========================================================================

import pandas as pd
import numpy as np

from calculadora import (
    Fase, simular, ESTRATEGIAS,
    continuar_de, validar_parametros,
)

TAXA_DECIMAL = TAXA_MENSAL_PCT / 100

In [ ]:
# =========================================================================
# SIMULACAO COM NOSSAS FUNCOES
# =========================================================================

registros = projetar_saldo(CAPITAL_INICIAL, APORTE_MENSAL,
                            TAXA_DECIMAL, PRAZO_MESES)
resumo = extrair_resumo(registros)
df_projecao = pd.DataFrame(registros)

print("--- Projecao (formulas proprias) ---")
print(f"Saldo final: R$ {resumo['saldo_final']:,.2f}")
print(f"Total investido: R$ {resumo['total_investido']:,.2f}")
print(f"Total em juros: R$ {resumo['total_juros']:,.2f} ({resumo['pct_juros']}%)")
print(f"Tempo: {resumo['meses']} meses ({resumo['anos']} anos)")

In [ ]:
# =========================================================================
# TAXAS EQUIVALENTES
# =========================================================================

taxas_conv = calcular_taxa_equivalente(TAXA_DECIMAL)
print("--- Taxas Equivalentes ---")
for periodo, taxa in taxas_conv.items():
    print(f"  {periodo.capitalize():>9}: {taxa * 100:.4f}%")

In [ ]:
# =========================================================================
# APORTE NECESSARIO PARA A META
# =========================================================================

aporte_30a = calcular_aporte_necessario(META, TAXA_DECIMAL, 360)
aporte_20a = calcular_aporte_necessario(META, TAXA_DECIMAL, 240)
aporte_10a = calcular_aporte_necessario(META, TAXA_DECIMAL, 120)

print("--- Aporte necessario para atingir a meta ---")
print(f"  Em 30 anos: R$ {aporte_30a:,.2f}/mes")
print(f"  Em 20 anos: R$ {aporte_20a:,.2f}/mes")
print(f"  Em 10 anos: R$ {aporte_10a:,.2f}/mes")

In [ ]:
# =========================================================================
# SIMULACAO AVANCADA COM MULTIPLAS FASES
# =========================================================================

fases = [
    Fase(meses=12,  aporte=APORTE_MENSAL * 0.5, nome="Inicio"),
    Fase(meses=24,  aporte=APORTE_MENSAL * 1.0, nome="Aceleracao"),
    Fase(meses=None, aporte=APORTE_MENSAL * 1.2, nome="Consolidacao"),
]

estrategia = ESTRATEGIAS.get(ESTRATEGIA)

if MES_INICIAL > 0 and CAPITAL_INICIAL > 0:
    resultado = continuar_de(TAXA_DECIMAL, META, fases,
                              MES_INICIAL, CAPITAL_INICIAL,
                              estrategia=estrategia)
else:
    resultado = simular(TAXA_DECIMAL, META, fases,
                         estrategia=estrategia,
                         saldo_inicial=CAPITAL_INICIAL,
                         mes_inicial=MES_INICIAL,
                         max_meses=PRAZO_MESES)

print("--- Simulacao Avancada (multiplas fases) ---")
if resultado.erro:
    print(f"Aviso: {resultado.erro}")
print(f"Saldo final: R$ {resultado.saldo_final:,.2f}")
print(f"Total investido: R$ {resultado.total_investido:,.2f}")
print(f"Total juros: R$ {resultado.total_juros:,.2f}")
print(f"Tempo: {resultado.anos}a {resultado.meses_resto}m")
print(f"Atingiu a meta: {'Sim' if resultado.atingiu_meta else 'Nao'}")

---
## Bloco 4 — Resultados e Visualização

Gráficos da evolução do montante ao longo do tempo.

In [ ]:
# =========================================================================
# CONFIGURACAO DOS GRAFICOS
# =========================================================================

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

In [ ]:
# =========================================================================
# GRAFICO 1 - EVOLUCAO DO PATRIMONIO (formulas proprias)
# =========================================================================

fig, ax = plt.subplots()

ax.plot(df_projecao["Mes"], df_projecao["Saldo"],
        label="Saldo acumulado", linewidth=2, color="#2196F3")
ax.plot(df_projecao["Mes"], df_projecao["Total_Investido"],
        label="Total aportado", linewidth=1.5, color="#4CAF50", linestyle="--")
ax.fill_between(df_projecao["Mes"], df_projecao["Total_Investido"],
                df_projecao["Saldo"], alpha=0.15, color="#2196F3",
                label="Juros compostos")

if META > 0:
    ax.axhline(y=META, color="#FF9800", linestyle="--", alpha=0.7,
               label=f"Meta: R$ {META:,.0f}")

ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"R$ {x/1000:,.0f}k"
                          if x >= 1000 else f"R$ {x:,.0f}"))
ax.set_xlabel("Meses")
ax.set_ylabel("Valor (R$)")
ax.set_title("Evolucao do Patrimonio (formulas proprias)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================================
# GRAFICO 2 - EVOLUCAO POR ANO (multiplas fases)
# =========================================================================

if resultado.df is not None and len(resultado.df) > 0:
    df_avancado = resultado.df

    fig, ax = plt.subplots()

    ax.plot(df_avancado["Ano"], df_avancado["Saldo"],
            label="Saldo acumulado", linewidth=2, color="#2196F3")
    ax.plot(df_avancado["Ano"], df_avancado["Aportes_Acum"],
            label="Total aportado", linewidth=1.5, color="#4CAF50", linestyle="--")
    ax.fill_between(df_avancado["Ano"], df_avancado["Aportes_Acum"],
                    df_avancado["Saldo"], alpha=0.15, color="#2196F3",
                    label="Juros compostos")

    ax.axhline(y=META, color="#FF9800", linestyle="--", alpha=0.7,
               label=f"Meta: R$ {META:,.0f}")

    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"R$ {x/1000:,.0f}k"
                              if x >= 1000 else f"R$ {x:,.0f}"))
    ax.set_xlabel("Anos")
    ax.set_ylabel("Valor (R$)")
    ax.set_title("Evolucao do Patrimonio (multiplas fases)")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

In [ ]:
# =========================================================================
# GRAFICO 3 - COMPOSICAO DO MONTANTE FINAL
# =========================================================================

if resumo:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    labels = ["Total investido", "Rendimentos (juros)"]
    valores = [resumo["total_investido"], resumo["total_juros"]]
    cores = ["#4CAF50", "#2196F3"]

    ax1.pie(valores, labels=labels, autopct="%1.1f%%",
            startangle=90, colors=cores, explode=(0, 0.05))
    ax1.set_title("Composicao do Montante Final")

    barras = ax2.bar(labels, valores, color=cores, width=0.5)
    offset = max(valores) * 0.02 if max(valores) > 0 else 100
    for barra, valor in zip(barras, valores):
        ax2.text(barra.get_x() + barra.get_width() / 2,
                 barra.get_height() + offset,
                 f"R$ {valor:,.2f}", ha="center",
                 fontsize=11, fontweight="bold")
    ax2.set_ylabel("Valor (R$)")
    ax2.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"R$ {x/1000:,.0f}k"
                              if x >= 1000 else f"R$ {x:,.0f}"))
    ax2.set_title("Total Investido vs Rendimentos")
    plt.tight_layout()
    plt.show()

In [ ]:
# =========================================================================
# TABELA - RESUMO DOS RESULTADOS
# =========================================================================

print("=" * 60)
print("RESUMO COMPLETO DA SIMULACAO")
print("=" * 60)
print()
print(f"Parametros:")
print(f"  Capital inicial: R$ {CAPITAL_INICIAL:,.2f}")
print(f"  Taxa mensal: {TAXA_MENSAL_PCT:.2f}%")
print(f"  Taxa anual: {taxas_conv['anual'] * 100:.2f}%")
print(f"  Aporte mensal: R$ {APORTE_MENSAL:,.2f}")
print(f"  Meta: R$ {META:,.2f}")
print(f"  Prazo maximo: {PRAZO_MESES} meses ({PRAZO_MESES // 12} anos)")
print()
print("Resultados (projecao simples):")
print(f"  Saldo final: R$ {resumo['saldo_final']:,.2f}")
print(f"  Total investido: R$ {resumo['total_investido']:,.2f}")
print(f"  Total em juros: R$ {resumo['total_juros']:,.2f}")
print(f"  Proporcao juros: {resumo['pct_juros']}%")
print()
print("Resultados (multiplas fases):")
print(f"  Saldo final: R$ {resultado.saldo_final:,.2f}")
print(f"  Total investido: R$ {resultado.total_investido:,.2f}")
print(f"  Total juros: R$ {resultado.total_juros:,.2f}")
print(f"  Tempo total: {resultado.anos}a {resultado.meses_resto}m")
print(f"  Atingiu a meta: {'Sim' if resultado.atingiu_meta else 'Nao'}")

In [ ]:
# =========================================================================
# TABELA MES A MES (amostra)
# =========================================================================

print("--- Projecao mes a mes (primeiras 10 linhas) ---")
display(df_projecao.head(10))

print(f"\nTotal de linhas: {len(df_projecao)}")

In [ ]:
# =========================================================================
# EXPORTACAO PARA CSV
# =========================================================================

if EXPORTAR_CSV:
    if resultado.df is not None:
        resultado.df.to_csv(ARQUIVO_CSV, index=False,
                            decimal=",", sep=";")
        print(f"CSV exportado: {ARQUIVO_CSV} ({len(resultado.df)} linhas)")
    else:
        print("Nenhum dado da simulacao avancada para exportar.")